# Green Claims — modular RAG
Open this notebook from a Databricks Git folder containing the **whole repository**.
The package is the source of truth; no application code is copied into this notebook.
For one-file import without a repository, use `mvp_demo/Green_Claims_Databricks.ipynb`.

Requires Python 3.10+. The first cell adds the repository `src` directory to the Python path.
Optional dependencies: `pypdf` for PDFs; `ipywidgets` and IPython for the interactive UI.
Install approved packages separately if your compute does not include them.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = ""  # Set /Workspace/.../cro_hackathon_greenwashing if auto-discovery fails.
if REPO_ROOT:
    root = Path(REPO_ROOT).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    root = next((p for p in candidates if (p / "src/green_claims").is_dir()), None)
if root is None or not (root / "src/green_claims").is_dir():
    raise ValueError("Set REPO_ROOT to the full repository path and rerun this cell.")
sys.path.insert(0, str(root / "src"))


## Configuration
Use names of existing Databricks secrets, never actual keys. These settings can remain empty for offline runs.

In [ ]:
import os
from green_claims.config import Settings

settings = Settings(
    api_url=os.getenv("AZURE_OPENAI_CHAT_URL", ""),  # Full chat/completions URL
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT", ""),  # Deployment name
    secret_scope="",  # Existing Databricks secret scope
    secret_key="",    # Secret entry NAME, not its value
    auth_type="api-key",
    language="English",
)


## Deterministic demo or live review
This cell runs without widgets. Select `Live AI` only after configuring approved model access.
Set `REGULATION_PATH` to replace the fictional standard with PDF/TXT/MD; leave it empty for the demo.

In [ ]:
from green_claims.demo.fixtures import load_cases, demo_regulation
from green_claims.interfaces.notebook import resolve_regulation
from green_claims.pipeline import run_review
from green_claims.reporting.html import render_report

CASE = "A"
MODE = "Offline demo"
VERIFICATION = "none"  # Live AI: "facts" or "facts_and_judge" adds model calls.
REGULATION_PATH = ""
REGULATION_VERSION = "Not specified"
case = next(c for c in load_cases() if c["id"] == CASE)
regulation = (resolve_regulation("Workspace file", REGULATION_VERSION, path=REGULATION_PATH)
              if REGULATION_PATH else demo_regulation())
client = settings.make_client(globals().get("dbutils")) if MODE == "Live AI" else None
report = run_review("\n".join(case["claims"]), case["disclosure"], regulation, MODE,
                    generator=client, model=settings.model, language=settings.language,
                    verification=VERIFICATION)
if "displayHTML" in globals():
    displayHTML(render_report(report))
else:
    print(report["summary"])


## Optional interactive view
The view supports edited inputs, Live AI, optional fact checks and LLM judge, regulation uploads, and file paths. After changing configuration, rerun this cell to create a view with the new settings.

In [ ]:
SHOW_UI = False  # Set True if ipywidgets is available in your runtime.
if SHOW_UI:
    from green_claims.interfaces.notebook import show_review
    controls = show_review(settings, dbutils=globals().get("dbutils"))
